# 1.1 Unknown

참고 노트북
- [1.0 의도분류](1_0_의도분류.ipynb)
- [0.3 Jev와 OpenAI 사용 방법](0_3_Jev와_LLM_사용_방법.ipynb)

`1_0`에서 고객의 목적을 구분하기 어려워 `unknown`으로 분류했다면, 고객에게 무엇을 원하는지 확인합니다.

- **입력:** 고객 문장과 `unknown` 의도·진행 여부·이전 대화가 담긴 `SessionState`
- **처리:** `generate_unknown_response(state)`가 서비스의 지침과 대화를 OpenAI에 전달
- **출력:** 고객에게 보여 줄 확인 질문과 실제 응답 모델명이 담긴 `SessionState`

프로젝트 루트(`main.py`가 있는 폴더)에서 실행합니다. `OPENAI_API_KEY` 환경변수가 필요하며, 연결 테스트는 실제 OpenAI API를 호출합니다.

```mermaid
flowchart LR
    input["Input<br/>SessionState"]
    generate["OpenAI 확인 질문 생성<br/>generate_unknown_response(state)"]
    output["Output<br/>SessionState"]
    input --> generate --> output
    classDef io fill:#eeeeee,stroke:#999999,color:#333333;
    class input,output io;
```

## 1. 스키마 설정

고객 입력, 분류 결과와 대화 상태는 FastAPI 공통 스키마를 사용합니다.

In [1]:
# 1. 경로 준비: 프로젝트 루트를 공통 스키마를 찾는 경로에 추가합니다.
import sys
import os
sys.path.append(os.getcwd())

# 2. 스키마 확인: 이번 실습에 필요한 자료형만 불러옵니다.
from schemas.session_state import CustomerInput, IntentResult, SessionState
print("불러온 스키마:", CustomerInput.__name__, IntentResult.__name__, SessionState.__name__)

불러온 스키마: CustomerInput IntentResult SessionState


## 2. 모델 준비

환경변수에 설정된 OpenAI 키와 모델명으로 클라이언트를 준비합니다.

In [2]:
# 1. 모델 준비: 환경변수에서 키와 모델명을 읽어 클라이언트를 만듭니다.
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
model = os.getenv("OPENAI_MODEL", "gpt-5-nano").strip() or "gpt-5-nano"
print("설정한 모델:", model)

설정한 모델: gpt-5-nano


## 3. 관련 프롬프트 불러오기

공통 말투와 확인 질문 지침을 파일에서 읽습니다.

In [3]:
# 1. 지침 읽기: 말투와 확인 질문 지침을 읽어 모델에 보낼 내용으로 묶습니다.
from pathlib import Path
persona = Path("prompts/persona.md").read_text(encoding="utf-8")
unknown_prompt = Path("prompts/unknown_response.md").read_text(encoding="utf-8")
prompt = f"{persona.strip()}\n\n{unknown_prompt.strip()}"
print(prompt)

# 챗봇 성격과 말투

당신은 모찌카페의 친절하고 차분한 안내 도우미다.

- 고객을 존중하며 자연스러운 한국어 존댓말을 사용한다.
- 고객이 궁금해하거나 요청한 내용에 집중하여 이해하기 쉽게 답한다.
- 답변은 필요한 내용 위주로 간결하게 작성한다.
- 과장된 표현이나 불필요한 감탄, 반복적인 인사는 피한다.
- 확인되지 않은 내용을 지어내지 않고, 필요한 경우 고객에게 확인한다.

# 역할

당신은 모찌카페의 안내 도우미다. 의도를 확실히 분류하지 못한 고객에게 다음 요청을 선택할 수 있도록 한국어로 답한다.

## 입력

- `text`: 고객의 현재 요청
- `has_image`: 사진 첨부 여부. 사진 원본과 내용은 전달되지 않는다.
- `session_state`: 진행 중인 작업 등 제공된 맥락
- `session_state.intent`: 이전에 확인한 작업 의도. 이력이 없으면 `null`이다.
- 이전 대화: 앞선 `user`와 `assistant` 메시지

## 응답 기준

- 고객의 요청과 제공된 맥락에 맞춰 어떤 도움이 필요한지 자연스러운 확인 질문을 한다.
- 이전 작업 의도가 없으면 메뉴 검색, 스탬프 적립, 메뉴 추천, 카페정보문의, 상품 주문 중 무엇을 도와드릴지 선택지를 제시한다. 없는 작업 이력을 가정하지 않는다.
- `session_state.intent`에 작업 의도가 있으면, 마지막 고객 입력인 `text`의 내용에는 답변할 수 없다고 먼저 안내한다. 그다음 이전 대화를 참고해 기존 작업을 계속할지 묻는다. 예를 들어 “지금 말씀하신 내용에는 답변해 드리기 어렵습니다. 이전에 진행하던 메뉴 추천을 계속할까요?”처럼 답한다. 고객이 동의하기 전에 이전 작업을 진행한다고 말하거나 추가 조건부터 묻지 않는다.
- 이번 요청은 분류가 보류된 상태다. `session_state.intent`는 이번 분류 결과가 아니라 유지된 기존 작업 의도이며, `null`이면 이전 작업이 없는 경우로 다룬다.
- 고객이 명확히 다른 카페 관련 

## 4. 사용자 input 준비

분류 결과가 `unknown`인 요청을 준비합니다.

In [4]:
# 1. 입력 준비: 여러 요청 중 어느 쪽을 먼저 처리할지 모르는 상태를 만듭니다.
state = SessionState(input=CustomerInput(text="메뉴 추천해주고 주차도 알려주세요"), intent=IntentResult(intent="unknown"))
print("고객 입력:", state.input.text)
print("의도:", state.intent.intent)

고객 입력: 메뉴 추천해주고 주차도 알려주세요
의도: unknown


## 5. 함수 만들기

고객 입력과 대화를 보내고, 생성된 확인 질문을 같은 상태에 담습니다.

In [5]:
# 1. 도구 준비: 상태를 모델에 전달할 JSON 문자열로 바꿉니다.
import json

def generate_unknown_response(state: SessionState):
    """고객 입력과 이전 대화로 확인 질문을 생성해 같은 상태에 담아 돌려줍니다."""
    # 1. 입력 준비: 지침과 이전 대화를 순서대로 담고 현재 입력을 마지막에 추가합니다.
    messages = [{"role": "system", "content": prompt}]
    messages.extend(message.model_dump(mode="json") for message in state.history)

    ## 현재 입력은 이전 대화 뒤에 별도 메시지로 붙입니다.
    current_input = {"text": state.input.text, "has_image": state.input.has_image, "previous_intent": state.previous_intent}
    messages.append({"role": "user", "content": json.dumps(current_input, ensure_ascii=False)})

    # 2. 모델 호출: 요청의 목적을 확인할 질문을 생성합니다.
    response = client.responses.create(model=model, input=messages)

    ## 생성된 답변이 비어 있으면 그대로 저장하지 않습니다.
    answer = response.output_text.strip()
    if not answer:
        raise ValueError("모델이 답변을 반환하지 않았습니다.")

    # 3. 결과 저장: 확인 질문과 실제 응답 모델명을 상태에 담습니다.
    state.answer = {"answer": answer, "model": response.model}

    return state

## 6. 연결 테스트

각 예제에서 입력을 만들고 함수를 호출해 확인 질문을 봅니다. 실제 OpenAI API를 호출합니다.

### 테스트 1 - 여러 요청

In [6]:
# 1. 입력 준비: 추천과 주차 문의가 함께 들어온 상태를 만듭니다.
state = SessionState(input=CustomerInput(text="메뉴 추천해주고 주차도 알려주세요"), intent=IntentResult(intent="unknown"))
print("Input:", state.input.text)
print("Intent:", state.intent.intent)

# 2. 함수 호출: 고객이 먼저 원하는 내용을 묻습니다.
print("\nFunction: generate_unknown_response(state)")
state = generate_unknown_response(state)

# 3. 결과 확인: 생성된 질문과 모델을 보여줍니다.
print("\nOutput:", state.answer["answer"])
print("Model:", state.answer["model"])

Input: 메뉴 추천해주고 주차도 알려주세요
Intent: unknown

Function: generate_unknown_response(state)



Output: 다음 중 어떤 도움을 원하실까요?
- 메뉴 검색
- 스탬프 적립
- 메뉴 추천
- 카페정보문의(주차 포함)
- 상품 주문

메뉴 추천과 주차 정보를 함께 드릴 수 있습니다. 어느 순서로 도와드릴지 알려 주세요.
Model: gpt-5-nano-2025-08-07


### 테스트 2 - 맥락 없는 답변

In [7]:
# 1. 입력 준비: 첫 대화에서 무엇에 대한 답인지 알 수 없는 문장을 준비합니다.
state = SessionState(input=CustomerInput(text="네, 맞아요"), intent=IntentResult(intent="unknown"))
print("Input:", state.input.text)
print("Intent:", state.intent.intent)

# 2. 함수 호출: 어떤 내용을 원하는지 확인합니다.
print("\nFunction: generate_unknown_response(state)")
state = generate_unknown_response(state)

# 3. 결과 확인: 생성된 질문과 모델을 보여줍니다.
print("\nOutput:", state.answer["answer"])
print("Model:", state.answer["model"])

Input: 네, 맞아요
Intent: unknown

Function: generate_unknown_response(state)



Output: 도와드릴 카페 관련 작업을 선택해 주세요. 1) 메뉴 검색 2) 스탬프 적립 3) 메뉴 추천 4) 카페 정보 문의 5) 상품 주문 중에서 어떤 것으로 도와드릴까요?
Model: gpt-5-nano-2025-08-07
